# Sanity Check - Step 04: Interpolate Bad Channels

Überprüft:
- Interpolation durchgeführt
- Bads-Liste geleert nach Interpolation
- Kanal-Anzahl gleich geblieben
- Amplituden noch plausibel

In [ ]:
import sys
from pathlib import Path
import mne

sys.path.append(str(Path.cwd().parent / 'eeg_pipeline'))
import config

print("Setup erfolgreich")

## 1. Before & After laden

In [ ]:
subject_id = config.SUBJECTS[0]

for person in ["P1", "P2"]:
    before_path = config.OUTPUT_DIR / f"sub-{subject_id}_{person}_badchannels_detected.fif"
    after_path = config.OUTPUT_DIR / f"sub-{subject_id}_{person}_interpolated.fif"
    
    if not before_path.exists():
        print(f"✗ {person}: Before-file not found")
        continue
    if not after_path.exists():
        print(f"✗ {person}: After-file not found")
        continue
    
    raw_before = mne.io.read_raw_fif(str(before_path), preload=False)
    raw_after = mne.io.read_raw_fif(str(after_path), preload=False)
    
    print(f"✓ {person}: Both files loaded")

## 2. Bad Channels Vergleich

In [ ]:
for person in ["P1", "P2"]:
    before_path = config.OUTPUT_DIR / f"sub-{subject_id}_{person}_badchannels_detected.fif"
    after_path = config.OUTPUT_DIR / f"sub-{subject_id}_{person}_interpolated.fif"
    
    if not before_path.exists() or not after_path.exists():
        continue
    
    raw_before = mne.io.read_raw_fif(str(before_path), preload=False)
    raw_after = mne.io.read_raw_fif(str(after_path), preload=False)
    
    bads_before = raw_before.info.get('bads', [])
    bads_after = raw_after.info.get('bads', [])
    
    print(f"\n=== {person} ===")
    print(f"Bad channels BEFORE interpolation: {len(bads_before)}")
    if bads_before:
        print(f"  {', '.join(bads_before)}")
    
    print(f"Bad channels AFTER interpolation: {len(bads_after)}")
    if bads_after:
        print(f"  {', '.join(bads_after)}")
    else:
        print(f"  ✓ Bad-Channels gelöscht nach Interpolation")

## 3. Metadata Überprüfung

In [ ]:
for person in ["P1", "P2"]:
    before_path = config.OUTPUT_DIR / f"sub-{subject_id}_{person}_badchannels_detected.fif"
    after_path = config.OUTPUT_DIR / f"sub-{subject_id}_{person}_interpolated.fif"
    
    if not before_path.exists() or not after_path.exists():
        continue
    
    raw_before = mne.io.read_raw_fif(str(before_path), preload=False)
    raw_after = mne.io.read_raw_fif(str(after_path), preload=False)
    
    print(f"\n{person}:")
    
    # Channel count
    if len(raw_before.ch_names) == len(raw_after.ch_names):
        print(f"  ✓ Kanal-Anzahl erhalten: {len(raw_after.ch_names)}")
    else:
        print(f"  ✗ Kanal-Anzahl geändert: {len(raw_before.ch_names)} -> {len(raw_after.ch_names)}")
    
    # Sampling rate
    if raw_before.info['sfreq'] == raw_after.info['sfreq']:
        print(f"  ✓ Sampling rate gleich: {raw_after.info['sfreq']} Hz")
    else:
        print(f"  ✗ Sampling rate geändert: {raw_before.info['sfreq']} -> {raw_after.info['sfreq']}")
    
    # Sample count
    if raw_before.n_times == raw_after.n_times:
        print(f"  ✓ Sample-Anzahl gleich: {raw_after.n_times}")
    else:
        print(f"  ✗ Sample-Anzahl geändert: {raw_before.n_times} -> {raw_after.n_times}")